# 01 - Data preparation

Build the modelling table for predicting `REFUSED` (loan application refusal).

- Seed = 42 everywhere
- Preprocessing fitted on train only
- OOT = `DAYS_DECISION > -365`, development data split 70/15/15 stratified
- Run cells in order, one at a time

In [1]:
# Cell A - setup + load previous_application with compact dtypes
import gc
import psutil
import numpy as np
import pandas as pd

SEED = 42
DATA = "../data/"
OUT = "../outputs/"
MODELS = "../models/"

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

def free_ram_gb():
    return round(psutil.virtual_memory().available / 1e9, 2)

def shrink(df):
    for c in df.columns:
        dt = df[c].dtype
        if pd.api.types.is_float_dtype(dt) and dt != np.float32:
            df[c] = df[c].astype(np.float32)
        elif pd.api.types.is_integer_dtype(dt):
            df[c] = pd.to_numeric(df[c], downcast="integer")
        elif pd.api.types.is_string_dtype(dt):
            df[c] = df[c].astype("category")
    return df

print("Free RAM before read (GB):", free_ram_gb())
prev = pd.read_csv(DATA + "previous_application.csv")
raw_mb = prev.memory_usage(deep=True).sum() / 1e6
prev = shrink(prev)
gc.collect()

print("previous_application shape:", prev.shape)
print("memory MB before -> after:", round(raw_mb), "->", round(prev.memory_usage(deep=True).sum() / 1e6))
print("Free RAM after read (GB):", free_ram_gb())
print()
print(prev.dtypes)

Free RAM before read (GB): 3.19


previous_application shape: (1670214, 37)
memory MB before -> after: 704 -> 154
Free RAM after read (GB): 2.39

SK_ID_PREV                        int32
SK_ID_CURR                        int32
NAME_CONTRACT_TYPE             category
AMT_ANNUITY                     float32
AMT_APPLICATION                 float32
AMT_CREDIT                      float32
AMT_DOWN_PAYMENT                float32
AMT_GOODS_PRICE                 float32
WEEKDAY_APPR_PROCESS_START     category
HOUR_APPR_PROCESS_START            int8
FLAG_LAST_APPL_PER_CONTRACT    category
NFLAG_LAST_APPL_IN_DAY             int8
RATE_DOWN_PAYMENT               float32
RATE_INTEREST_PRIMARY           float32
RATE_INTEREST_PRIVILEGED        float32
NAME_CASH_LOAN_PURPOSE         category
NAME_CONTRACT_STATUS           category
DAYS_DECISION                     int16
NAME_PAYMENT_TYPE              category
CODE_REJECT_REASON             category
NAME_TYPE_SUITE                category
NAME_CLIENT_TYPE               category
NAME_GOO

In [2]:
# Cell B - build target REFUSED (Approved=0, Refused=1); drop Canceled / Unused offer
rows_before = len(prev)

prev["REFUSED"] = (prev["NAME_CONTRACT_STATUS"] == "Refused").astype(np.int8)

print("NAME_CONTRACT_STATUS counts (original):")
print(prev["NAME_CONTRACT_STATUS"].value_counts())

prev = prev[prev["NAME_CONTRACT_STATUS"].isin(["Approved", "Refused"])].copy()
prev.reset_index(drop=True, inplace=True)

print()
print("rows before -> after:", rows_before, "->", len(prev),
      "(removed", rows_before - len(prev), "rows)")
print("REFUSED dtype:", prev["REFUSED"].dtype)
print()
print("class counts:")
print(prev["REFUSED"].value_counts().sort_index())
print("class %:")
print((prev["REFUSED"].value_counts(normalize=True).sort_index() * 100).round(2))
print("refusal rate: %.2f%%" % (prev["REFUSED"].mean() * 100))
print("Free RAM (GB):", free_ram_gb())

NAME_CONTRACT_STATUS counts (original):
NAME_CONTRACT_STATUS
Approved        1036781
Canceled         316319
Refused          290678
Unused offer      26436
Name: count, dtype: int64



rows before -> after: 1670214 -> 1327459 (removed 342755 rows)
REFUSED dtype: int8

class counts:
REFUSED
0    1036781
1     290678
Name: count, dtype: int64
class %:
REFUSED
0    78.1
1    21.9
Name: proportion, dtype: float64
refusal rate: 21.90%
Free RAM (GB): 2.43


In [3]:
# Cell C - leakage screen: is a column filled only AFTER the lender decides?
# If missing% differs hugely between approved and refused rows, that column cannot be
# known at application time -> it leaks the answer.
cands = [
    "AMT_ANNUITY", "AMT_APPLICATION", "AMT_CREDIT", "AMT_DOWN_PAYMENT",
    "AMT_GOODS_PRICE", "RATE_DOWN_PAYMENT", "RATE_INTEREST_PRIMARY",
    "RATE_INTEREST_PRIVILEGED", "CNT_PAYMENT", "NAME_PAYMENT_TYPE",
    "CODE_REJECT_REASON", "NAME_YIELD_GROUP", "PRODUCT_COMBINATION",
    "NFLAG_INSURED_ON_APPROVAL", "DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE",
    "DAYS_LAST_DUE_1ST_VERSION", "DAYS_LAST_DUE", "DAYS_TERMINATION",
]

ref = prev["REFUSED"]
rows = []
for c in cands:
    s = prev[c]
    miss0 = s[ref == 0].isna().mean() * 100
    miss1 = s[ref == 1].isna().mean() * 100
    pct_365 = (s == 365243).mean() * 100 if pd.api.types.is_numeric_dtype(s.dtype) else np.nan
    rows.append({"column": c,
                 "miss%_approved": round(float(miss0), 1),
                 "miss%_refused": round(float(miss1), 1),
                 "gap": round(float(miss0 - miss1), 1),
                 "n_unique": int(s.nunique(dropna=True)),
                 "%_eq_365243": round(float(pct_365), 1) if pct_365 == pct_365 else np.nan})

screen = pd.DataFrame(rows).set_index("column").sort_values("gap", ascending=False)
print("missingness by target (gap = approved - refused)")
print(screen)
print()
print("CODE_REJECT_REASON values:")
print(prev["CODE_REJECT_REASON"].value_counts())
print()
print("Free RAM (GB):", free_ram_gb())

missingness by target (gap = approved - refused)
                           miss%_approved  miss%_refused   gap  n_unique  %_eq_365243
column                                                                               
AMT_APPLICATION                       0.0            0.0   0.0     92737          0.0
AMT_CREDIT                            0.0            0.0   0.0     85266          0.0
NAME_PAYMENT_TYPE                     0.0            0.0   0.0         4          NaN
PRODUCT_COMBINATION                   0.0            0.0  -0.0        17          NaN
CODE_REJECT_REASON                    0.0            0.0   0.0         8          NaN
NAME_YIELD_GROUP                      0.0            0.0   0.0         5          NaN
RATE_INTEREST_PRIMARY                99.4          100.0  -0.6       148          0.0
RATE_INTEREST_PRIVILEGED             99.4          100.0  -0.6        25          0.0
AMT_GOODS_PRICE                       4.1           12.8  -8.6     92737          0.0
AMT_A

In [4]:
# Cell D - two checks before we freeze the drop list

print("mean amounts by target:")
print(prev.groupby("REFUSED")[["AMT_APPLICATION", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE"]].mean().round(0))

print()
print("% rows where AMT_CREDIT == AMT_APPLICATION, by target:")
eq = (prev["AMT_CREDIT"] == prev["AMT_APPLICATION"]).groupby(prev["REFUSED"]).mean() * 100
print(eq.round(2))

print()
print("CODE_REJECT_REASON x REFUSED (row counts):")
print(pd.crosstab(prev["CODE_REJECT_REASON"], prev["REFUSED"]))

print()
print("Free RAM (GB):", free_ram_gb())

mean amounts by target:
         AMT_APPLICATION  AMT_CREDIT  AMT_ANNUITY  AMT_GOODS_PRICE
REFUSED                                                           
0               180567.0    202564.0      14720.0         188349.0
1               331760.0    371690.0      20498.0         380453.0

% rows where AMT_CREDIT == AMT_APPLICATION, by target:
REFUSED
0    18.99
1    41.59
dtype: float64

CODE_REJECT_REASON x REFUSED (row counts):


REFUSED                   0       1
CODE_REJECT_REASON                 
HC                        0  175231
LIMIT                     0   55680
SCO                       0   37467
SCOFR                     0   12811
SYSTEM                    0     717
VERIF                     0    3535
XAP                 1036773       1
XNA                       8    5236

Free RAM (GB): 2.4


In [5]:
# Cell E - apply the leakage drop list
drop_cols = [
    "NAME_CONTRACT_STATUS", "CODE_REJECT_REASON", "AMT_CREDIT",
    "AMT_ANNUITY", "CNT_PAYMENT", "RATE_DOWN_PAYMENT",
    "RATE_INTEREST_PRIMARY", "RATE_INTEREST_PRIVILEGED",
    "NFLAG_INSURED_ON_APPROVAL",
    "DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE", "DAYS_LAST_DUE_1ST_VERSION",
    "DAYS_LAST_DUE", "DAYS_TERMINATION",
]
dropped = [c for c in drop_cols if c in prev.columns]
prev = prev.drop(columns=dropped)
gc.collect()
print("dropped", len(dropped), ":", dropped)
print("remaining columns:", len(prev.columns))
print(list(prev.columns))
print("memory MB:", round(prev.memory_usage(deep=True).sum() / 1e6))
print("Free RAM (GB):", free_ram_gb())

dropped 14 : ['NAME_CONTRACT_STATUS', 'CODE_REJECT_REASON', 'AMT_CREDIT', 'AMT_ANNUITY', 'CNT_PAYMENT', 'RATE_DOWN_PAYMENT', 'RATE_INTEREST_PRIMARY', 'RATE_INTEREST_PRIVILEGED', 'NFLAG_INSURED_ON_APPROVAL', 'DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE_1ST_VERSION', 'DAYS_LAST_DUE', 'DAYS_TERMINATION']
remaining columns: 24
['SK_ID_PREV', 'SK_ID_CURR', 'NAME_CONTRACT_TYPE', 'AMT_APPLICATION', 'AMT_DOWN_PAYMENT', 'AMT_GOODS_PRICE', 'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START', 'FLAG_LAST_APPL_PER_CONTRACT', 'NFLAG_LAST_APPL_IN_DAY', 'NAME_CASH_LOAN_PURPOSE', 'DAYS_DECISION', 'NAME_PAYMENT_TYPE', 'NAME_TYPE_SUITE', 'NAME_CLIENT_TYPE', 'NAME_GOODS_CATEGORY', 'NAME_PORTFOLIO', 'NAME_PRODUCT_TYPE', 'CHANNEL_TYPE', 'SELLERPLACE_AREA', 'NAME_SELLER_INDUSTRY', 'NAME_YIELD_GROUP', 'PRODUCT_COMBINATION', 'REFUSED']
memory MB: 57
Free RAM (GB): 2.39


In [6]:
# Cell F - load customer table (application_train) with the same shrink()
cust = pd.read_csv(DATA + "application_train.csv")
raw_mb_cust = cust.memory_usage(deep=True).sum() / 1e6
cust = shrink(cust)
gc.collect()

print("application_train shape:", cust.shape)
print("memory MB before -> after:", round(raw_mb_cust), "->", round(cust.memory_usage(deep=True).sum() / 1e6))
print("Free RAM (GB):", free_ram_gb())

overlap = sorted((set(prev.columns) & set(cust.columns)) - {"SK_ID_CURR", "REFUSED"})
print()
print("columns in BOTH tables (renamed CUST_* on the customer side):")
print(overlap)

application_train shape: (307511, 121)
memory MB before -> after: 339 -> 100
Free RAM (GB): 2.33

columns in BOTH tables (renamed CUST_* on the customer side):
['AMT_GOODS_PRICE', 'HOUR_APPR_PROCESS_START', 'NAME_CONTRACT_TYPE', 'NAME_TYPE_SUITE', 'WEEKDAY_APPR_PROCESS_START']


In [7]:
# Cell G - rename overlapping columns on the customer side, then LEFT merge
overlap = sorted((set(prev.columns) & set(cust.columns)) - {"SK_ID_CURR", "REFUSED"})
rename_map = {c: "CUST_" + c for c in overlap}
cust = cust.rename(columns=rename_map)

in_cust = prev["SK_ID_CURR"].isin(cust["SK_ID_CURR"])
print("renamed on customer side:", rename_map)
print("prev rows whose customer exists in application_train: %d of %d (%.1f%%)"
      % (in_cust.sum(), len(prev), in_cust.mean() * 100))

rows_before = len(prev)
prev = prev.merge(cust, on="SK_ID_CURR", how="left")
del cust, in_cust
gc.collect()

print()
print("rows before -> after merge:", rows_before, "->", len(prev), "(left join: must be identical)")
print("columns after merge:", prev.shape[1], " (expected 24 + 121 - 1 = 144)")
print("memory MB:", round(prev.memory_usage(deep=True).sum() / 1e6))
print("Free RAM (GB):", free_ram_gb())

renamed on customer side: {'AMT_GOODS_PRICE': 'CUST_AMT_GOODS_PRICE', 'HOUR_APPR_PROCESS_START': 'CUST_HOUR_APPR_PROCESS_START', 'NAME_CONTRACT_TYPE': 'CUST_NAME_CONTRACT_TYPE', 'NAME_TYPE_SUITE': 'CUST_NAME_TYPE_SUITE', 'WEEKDAY_APPR_PROCESS_START': 'CUST_WEEKDAY_APPR_PROCESS_START'}
prev rows whose customer exists in application_train: 1131489 of 1327459 (85.2%)



rows before -> after merge: 1327459 -> 1327459 (left join: must be identical)
columns after merge: 144  (expected 24 + 121 - 1 = 144)
memory MB: 838
Free RAM (GB): 1.62


In [8]:
# Cell H - OOT by time, then 70/15/15 stratified on development (seed 42)
# We split INDEX objects, not DataFrames, so we never copy the 838 MB table.
from sklearn.model_selection import train_test_split

is_oot = prev["DAYS_DECISION"] > -365
dev_idx = prev.index[~is_oot]
oot_idx = prev.index[is_oot]

print("OOT  (DAYS_DECISION > -365): %7d rows, refusal %.2f%%"
      % (len(oot_idx), prev.loc[oot_idx, "REFUSED"].mean() * 100))
print("DEV  (DAYS_DECISION <= -365): %7d rows, refusal %.2f%%"
      % (len(dev_idx), prev.loc[dev_idx, "REFUSED"].mean() * 100))

tr_idx, rest_idx = train_test_split(dev_idx, test_size=0.30,
                                    stratify=prev.loc[dev_idx, "REFUSED"], random_state=SEED)
va_idx, te_idx = train_test_split(rest_idx, test_size=0.50,
                                  stratify=prev.loc[rest_idx, "REFUSED"], random_state=SEED)

prev["SPLIT"] = "oot"
prev.loc[tr_idx, "SPLIT"] = "train"
prev.loc[va_idx, "SPLIT"] = "val"
prev.loc[te_idx, "SPLIT"] = "test"

for name in ["train", "val", "test", "oot"]:
    g = prev[prev["SPLIT"] == name]
    print("%-5s rows=%7d  (%.1f%%)  refusal=%.2f%%  unique customers=%d"
          % (name, len(g), len(g) / len(prev) * 100, g["REFUSED"].mean() * 100, g["SK_ID_CURR"].nunique()))

sets = {k: set(g["SK_ID_PREV"]) for k, g in prev.groupby("SPLIT", observed=True)}
disjoint = not (sets["train"] & sets["val"]) and not (sets["train"] & sets["test"]) and not (sets["val"] & sets["test"])
print("no SK_ID_PREV shared between train/val/test:", disjoint)
print("Free RAM (GB):", free_ram_gb())

OOT  (DAYS_DECISION > -365):  306791 rows, refusal 37.26%
DEV  (DAYS_DECISION <= -365): 1020668 rows, refusal 17.28%


C:\Users\anand\AppData\Local\Temp\ipykernel_9976\2987523477.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  prev["SPLIT"] = "oot"


train rows= 714467  (53.8%)  refusal=17.28%  unique customers=277398
val   rows= 153100  (11.5%)  refusal=17.28%  unique customers=114772


test  rows= 153101  (11.5%)  refusal=17.28%  unique customers=114763


oot   rows= 306791  (23.1%)  refusal=37.26%  unique customers=164239


no SK_ID_PREV shared between train/val/test: True
Free RAM (GB): 2.07


In [9]:
# Cell I - save the modelling table and prove it round-trips
import os

prev = prev.copy()  # de-fragment the frame after the merge inserts
gc.collect()

out_path = OUT + "stage1_modeling.parquet"
prev.to_parquet(out_path, index=False)
print("saved:", out_path, "| size MB:", round(os.path.getsize(out_path) / 1e6, 1))

check = pd.read_parquet(out_path)
print("read back shape:", check.shape)
print("columns match:", list(check.columns) == list(prev.columns))
print("dtypes match:", bool((check.dtypes == prev.dtypes).all()))
print("refusal rate in file: %.2f%%" % (check["REFUSED"].mean() * 100))
print("SPLIT counts:")
print(check["SPLIT"].value_counts())
del check
gc.collect()
print("Free RAM (GB):", free_ram_gb())

saved: ../outputs/stage1_modeling.parquet | size MB: 115.4


read back shape: (1327459, 145)
columns match: True
dtypes match: True
refusal rate in file: 21.90%
SPLIT counts:
SPLIT
train    714467
oot      306791
test     153101
val      153100
Name: count, dtype: int64
Free RAM (GB): 0.59


In [10]:
# Cell J - prefix EVERY customer-table column with CUST_ so names are unambiguous
# After Cell G only the 5 colliding names were prefixed. Because the application-level
# AMT_CREDIT / AMT_ANNUITY were dropped in Cell E, the CUSTOMER versions of those columns
# arrived from application_train unprefixed - which is exactly the ambiguity we want to kill.
# Rule: keep the 25 known application-level names, prefix everything else.
APP_LEVEL = {
    "SK_ID_PREV", "SK_ID_CURR", "NAME_CONTRACT_TYPE", "AMT_APPLICATION", "AMT_DOWN_PAYMENT",
    "AMT_GOODS_PRICE", "WEEKDAY_APPR_PROCESS_START", "HOUR_APPR_PROCESS_START",
    "FLAG_LAST_APPL_PER_CONTRACT", "NFLAG_LAST_APPL_IN_DAY", "NAME_CASH_LOAN_PURPOSE",
    "DAYS_DECISION", "NAME_PAYMENT_TYPE", "NAME_TYPE_SUITE", "NAME_CLIENT_TYPE",
    "NAME_GOODS_CATEGORY", "NAME_PORTFOLIO", "NAME_PRODUCT_TYPE", "CHANNEL_TYPE",
    "SELLERPLACE_AREA", "NAME_SELLER_INDUSTRY", "NAME_YIELD_GROUP", "PRODUCT_COMBINATION",
    "REFUSED", "SPLIT",
}
to_rename = {c: "CUST_" + c for c in prev.columns if c not in APP_LEVEL and not c.startswith("CUST_")}
prev = prev.rename(columns=to_rename)
gc.collect()

prev.to_parquet(OUT + "stage1_modeling.parquet", index=False)
print("renamed", len(to_rename), "customer columns, e.g.:", list(to_rename.items())[:6])
print("total columns:", prev.shape[1])
print("application-level columns (must stay unprefixed):")
print([c for c in prev.columns if not c.startswith("CUST_")])

check = pd.read_parquet(OUT + "stage1_modeling.parquet")
print("read back shape:", check.shape, "| columns match:", list(check.columns) == list(prev.columns))
print("refusal rate: %.2f%%" % (check["REFUSED"].mean() * 100))
print("SPLIT counts:")
print(check["SPLIT"].value_counts())
del check
gc.collect()
print("Free RAM (GB):", free_ram_gb())


renamed 115 customer columns, e.g.: [('CODE_GENDER', 'CUST_CODE_GENDER'), ('FLAG_OWN_CAR', 'CUST_FLAG_OWN_CAR'), ('FLAG_OWN_REALTY', 'CUST_FLAG_OWN_REALTY'), ('CNT_CHILDREN', 'CUST_CNT_CHILDREN'), ('AMT_INCOME_TOTAL', 'CUST_AMT_INCOME_TOTAL'), ('AMT_CREDIT', 'CUST_AMT_CREDIT')]
total columns: 145
application-level columns (must stay unprefixed):
['SK_ID_PREV', 'SK_ID_CURR', 'NAME_CONTRACT_TYPE', 'AMT_APPLICATION', 'AMT_DOWN_PAYMENT', 'AMT_GOODS_PRICE', 'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START', 'FLAG_LAST_APPL_PER_CONTRACT', 'NFLAG_LAST_APPL_IN_DAY', 'NAME_CASH_LOAN_PURPOSE', 'DAYS_DECISION', 'NAME_PAYMENT_TYPE', 'NAME_TYPE_SUITE', 'NAME_CLIENT_TYPE', 'NAME_GOODS_CATEGORY', 'NAME_PORTFOLIO', 'NAME_PRODUCT_TYPE', 'CHANNEL_TYPE', 'SELLERPLACE_AREA', 'NAME_SELLER_INDUSTRY', 'NAME_YIELD_GROUP', 'PRODUCT_COMBINATION', 'REFUSED', 'SPLIT']


read back shape: (1327459, 145) | columns match: True
refusal rate: 21.90%
SPLIT counts:
SPLIT
train    714467
oot      306791
test     153101
val      153100
Name: count, dtype: int64
Free RAM (GB): 0.76


### Step 4 - treat odd values, convert dates to readable units

Three data-quality issues from the dictionary and the profile:

- **`365243`** in `DAYS_*` columns is a placeholder for "no date", not a real day count -> set to missing.
- **`XNA` / `XAP`** in text columns mean "not available" / "not applicable" -> set to missing, so the model sees an honest gap instead of a fake category.
- **`DAYS_BIRTH` / `DAYS_EMPLOYED`** are negative day counts -> convert to `AGE_YEARS` and `EMPLOYED_YEARS` (positive years), which is what the spec asks for and what a human reader understands.

`DAYS_DECISION` is deliberately left untouched: it defines the out-of-time split.

In [11]:
# Cell K - treat odd values (365243, XNA/XAP), convert age and employment to years
days_cols = [c for c in prev.columns if "DAYS" in c.upper()]
before_365 = {}
for c in days_cols:
    if pd.api.types.is_numeric_dtype(prev[c]):
        n = int((prev[c] == 365243).sum())
        if n:
            before_365[c] = n
for c in before_365:
    prev[c] = prev[c].mask(prev[c] == 365243).astype("float32")

xna_before = {}
for c in prev.columns:
    if prev[c].dtype.name == "category":
        levels = set(map(str, prev[c].cat.categories)) & {"XNA", "XAP"}
        if levels:
            xna_before[c] = int(prev[c].isin(["XNA", "XAP"]).sum())
            prev[c] = prev[c].cat.remove_categories(sorted(levels))

prev["AGE_YEARS"] = (-prev["CUST_DAYS_BIRTH"] / 365.25).round(1).astype("float32")
prev["EMPLOYED_YEARS"] = (-prev["CUST_DAYS_EMPLOYED"] / 365.25).round(1).astype("float32")
prev = prev.drop(columns=["CUST_DAYS_BIRTH", "CUST_DAYS_EMPLOYED"])

print("365243 placeholder removed from:", before_365)
print("XNA/XAP cells removed:", sum(xna_before.values()), "across", len(xna_before), "columns")
print("DAYS_DECISION untouched:", int(prev["DAYS_DECISION"].min()), "to", int(prev["DAYS_DECISION"].max()))
print("AGE_YEARS      : %.1f - %.1f | missing %.2f%%"
      % (prev["AGE_YEARS"].min(), prev["AGE_YEARS"].max(), prev["AGE_YEARS"].isna().mean() * 100))
print("EMPLOYED_YEARS : %.1f - %.1f | missing %.2f%%"
      % (prev["EMPLOYED_YEARS"].min(), prev["EMPLOYED_YEARS"].max(), prev["EMPLOYED_YEARS"].isna().mean() * 100))
print("shape:", prev.shape, "| free RAM (GB):", free_ram_gb())

365243 placeholder removed from: {'CUST_DAYS_EMPLOYED': 213982}
XNA/XAP cells removed: 3921527 across 11 columns
DAYS_DECISION untouched: -2922 to -1
AGE_YEARS      : 20.5 - 69.0 | missing 14.76%
EMPLOYED_YEARS : 0.0 - 49.0 | missing 30.88%
shape: (1327459, 145) | free RAM (GB): 1.24


In [12]:
# Cell L - 11 engineered features (spec Part 2: >= 8 features, each justified in the markdown above)
prev["REQUEST_INCOME"] = (prev["AMT_APPLICATION"] / prev["CUST_AMT_INCOME_TOTAL"]).astype("float32")
prev["GOODS_FINANCED_PCT"] = (prev["AMT_APPLICATION"] / prev["AMT_GOODS_PRICE"]).astype("float32")
for c in ["REQUEST_INCOME", "GOODS_FINANCED_PCT"]:
    prev[c] = prev[c].replace([np.inf, -np.inf], np.nan)

prev["AGE_BAND"] = pd.cut(prev["AGE_YEARS"], bins=[20, 25, 30, 35, 40, 45, 50, 55, 60, 70],
    labels=["20-25", "25-30", "30-35", "35-40", "40-45", "45-50", "50-55", "55-60", "60+"])
prev["EMPLOYMENT_BAND"] = pd.cut(prev["EMPLOYED_YEARS"], bins=[-1, 2, 5, 10, 20, 50],
    labels=["<2y", "2-5y", "5-10y", "10-20y", "20y+"])
# income deciles computed on TRAIN rows only (preprocessing rule)
q = np.unique(prev.loc[prev["SPLIT"] == "train", "CUST_AMT_INCOME_TOTAL"]
              .quantile(np.linspace(0, 1, 11)).to_numpy())
prev["INCOME_BAND"] = pd.cut(prev["CUST_AMT_INCOME_TOTAL"], bins=q,
    labels=["Q%d" % i for i in range(1, len(q))], include_lowest=True)

ext = ["CUST_EXT_SOURCE_1", "CUST_EXT_SOURCE_2", "CUST_EXT_SOURCE_3"]
prev["EXT_MISSING_COUNT"] = prev[ext].isna().sum(axis=1).astype(np.int8)
prev["EXT_MEAN"] = prev[ext].mean(axis=1).astype("float32")

prev["WEEKEND_FLAG"] = prev["WEEKDAY_APPR_PROCESS_START"].isin(["SATURDAY", "SUNDAY"]).astype("int8")
prev["HOUR_BAND"] = pd.cut(prev["HOUR_APPR_PROCESS_START"], bins=[-1, 6, 12, 18, 24],
    labels=["00-06", "06-12", "12-18", "18-24"])
prev["DOWN_PAYMENT_FLAG"] = prev["AMT_DOWN_PAYMENT"].notna().astype("int8")
prev["PURPOSE_STATED"] = prev["NAME_CASH_LOAN_PURPOSE"].notna().astype("int8")

print(prev[["REQUEST_INCOME", "GOODS_FINANCED_PCT", "EXT_MEAN"]]
      .describe().T[["count", "mean", "min", "max"]].to_string())
for c in ["AGE_BAND", "EMPLOYMENT_BAND", "INCOME_BAND", "HOUR_BAND",
          "EXT_MISSING_COUNT", "WEEKEND_FLAG", "DOWN_PAYMENT_FLAG", "PURPOSE_STATED"]:
    print(c, "->", dict(prev[c].value_counts(dropna=False).head(6)))
print("inf check:", int(np.isinf(prev[["REQUEST_INCOME", "GOODS_FINANCED_PCT"]].to_numpy()).sum()))
print("shape:", prev.shape, "| free RAM (GB):", free_ram_gb())

                        count      mean       min        max
REQUEST_INCOME      1131489.0  1.361488  0.000000  87.719299
GOODS_FINANCED_PCT  1241147.0  1.000008  0.000000   9.000000
EXT_MEAN            1131227.0  0.502356  0.000006   0.863816
AGE_BAND -> {nan: np.int64(195970), '35-40': np.int64(157438), '40-45': np.int64(154480), '30-35': np.int64(144091), '60+': np.int64(141891), '45-50': np.int64(133200)}
EMPLOYMENT_BAND -> {nan: np.int64(409952), '2-5y': np.int64(273393), '5-10y': np.int64(254680), '<2y': np.int64(199079), '10-20y': np.int64(145160), '20y+': np.int64(45195)}
INCOME_BAND -> {nan: np.int64(195972), 'Q1': np.int64(186660), 'Q4': np.int64(168602), 'Q2': np.int64(150508), 'Q5': np.int64(128003), 'Q6': np.int64(125360)}
HOUR_BAND -> {'06-12': np.int64(631546), '12-18': np.int64(607062), '18-24': np.int64(46491), '00-06': np.int64(42360)}
EXT_MISSING_COUNT -> {1: np.int64(596508), 0: np.int64(417918), 3: np.int64(196232), 2: np.int64(116801)}
WEEKEND_FLAG -> {0: np.int64

### Engineered features (spec Part 2)

| Feature | Formula | Business justification |
|---|---|---|
| `REQUEST_INCOME` | `AMT_APPLICATION / CUST_AMT_INCOME_TOTAL` | Affordability: how many times annual income the customer is asking for. The single most direct repayment-capacity signal. |
| `GOODS_FINANCED_PCT` | `AMT_APPLICATION / AMT_GOODS_PRICE` | Share of the goods price financed. Near 100% = zero own contribution, which lenders treat as higher risk. |
| `AGE_BAND` | binned `AGE_YEARS` (20-25 ... 60+) | Refusal risk is not linear in age; bands let the model learn a U-shape without forcing one. |
| `EMPLOYMENT_BAND` | binned `EMPLOYED_YEARS` (<2y ... 20y+) | Job stability: very short tenure is the classic thin-file risk case; bands make the steps learnable. |
| `INCOME_BAND` | train-set income deciles (Q1-Q10) | Positions the applicant in the income distribution. Decile edges are computed on **train only**, per the preprocessing rule. |
| `EXT_MISSING_COUNT` | count of missing `CUST_EXT_SOURCE_1/2/3` | Missing credit bureau scores usually mean "no bureau history" - thin file is itself a risk signal. |
| `EXT_MEAN` | mean of available `CUST_EXT_SOURCE_*` | The external score is the strongest generic creditworthiness proxy in the data; averaging reduces single-source noise. |
| `WEEKEND_FLAG` | application on Sat/Sun | Applications submitted outside working hours are a different channel (online/impulsive) with a different refusal pattern. |
| `HOUR_BAND` | `HOUR_APPR_PROCESS_START` binned 00-06/06-12/12-18/18-24 | Time of day proxies the application channel (branch staffed hours vs overnight online). |
| `DOWN_PAYMENT_FLAG` | `AMT_DOWN_PAYMENT` is not recorded | Recording a down-payment proposal happens only for certain products; the flag separates those cases instead of hiding them in NaN. |
| `PURPOSE_STATED` | `NAME_CASH_LOAN_PURPOSE` still filled after XNA/XAP removal | 95% of purposes are blank after cleaning; a "was a real purpose stated" flag preserves the information content of the missing pattern. |

No engineered feature uses a column from the leakage drop list.

### Feature categorisation (spec Part 1 deliverable)

| Category | Examples in the model |
|---|---|
| **Demographic** (customer) | `AGE_YEARS`, `AGE_BAND`, `EMPLOYED_YEARS`, `EMPLOYMENT_BAND`, `CUST_CODE_GENDER`, `CUST_NAME_FAMILY_STATUS`, `CUST_CNT_CHILDREN`, `CUST_CNT_FAM_MEMBERS`, `CUST_REGION_RATING_CLIENT` |
| **Financial** (customer) | `CUST_AMT_INCOME_TOTAL`, `INCOME_BAND`, `CUST_AMT_CREDIT`, `CUST_AMT_ANNUITY`, `CUST_EXT_SOURCE_1/2/3`, `EXT_MEAN`, `EXT_MISSING_COUNT` |
| **Product** (application) | `NAME_CONTRACT_TYPE`, `NAME_PORTFOLIO`, `NAME_PRODUCT_TYPE`, `PRODUCT_COMBINATION`, `NAME_GOODS_CATEGORY`, `NAME_YIELD_GROUP`, `NAME_PAYMENT_TYPE`, `AMT_APPLICATION`, `AMT_DOWN_PAYMENT`, `AMT_GOODS_PRICE`, `REQUEST_INCOME`, `GOODS_FINANCED_PCT`, `DOWN_PAYMENT_FLAG` |
| **Channel** (application) | `CHANNEL_TYPE`, `NAME_SELLER_INDUSTRY`, `NAME_CLIENT_TYPE`, `NAME_TYPE_SUITE`, `WEEKDAY_APPR_PROCESS_START`, `WEEKEND_FLAG`, `HOUR_APPR_PROCESS_START`, `HOUR_BAND` |
| **History / behaviour** | `NAME_CASH_LOAN_PURPOSE`, `PURPOSE_STATED`, `FLAG_LAST_APPL_PER_CONTRACT`, `NFLAG_LAST_APPL_IN_DAY`, `SELLERPLACE_AREA`, customer flags (`CUST_FLAG_*`), customer history days (`CUST_DAYS_*`) |

Target, identifiers and the split marker (`REFUSED`, `SK_ID_PREV`, `SK_ID_CURR`, `SPLIT`, `DAYS_DECISION`) are excluded from the feature set.

### Save the model-ready table and build the star-schema data model

1. Data-quality summary (missing % and the action taken) and a duplicate check on the key.
2. Save `outputs/stage1_modeling.parquet` - the model-ready table the modelling notebook loads.
3. Star schema for Power BI: `fact_applications` + 4 dimension tables (`dim_client`, `dim_product`, `dim_channel`, `dim_date`), each with an integer key that the fact table references.

In [13]:
# Cell M - data quality summary, duplicate check, save the model-ready table
miss_pct = (prev.isna().mean() * 100).round(2)
miss_pct = miss_pct[miss_pct > 0].sort_values(ascending=False)

def action_for(col, pct):
    if col in ("NAME_CASH_LOAN_PURPOSE", "NAME_PRODUCT_TYPE", "NAME_GOODS_CATEGORY"):
        return "missing after XNA/XAP removal; PURPOSE_STATED flag added where useful"
    if col in ("EMPLOYED_YEARS", "CUST_DAYS_EMPLOYED", "CUST_ORGANIZATION_TYPE"):
        return "365243 placeholder or no customer match; model handles NaN"
    if col.startswith("CUST_") or col in ("AGE_YEARS",):
        return "no customer match on the join; kept as NaN, model handles it"
    if prev[col].dtype.name == "category":
        return "kept as a real category level; LightGBM splits on missing natively"
    return "kept as NaN; amount was simply not recorded for this row"

dq = pd.DataFrame({"missing_pct": miss_pct,
                   "action": [action_for(c, p) for c, p in miss_pct.items()]})
print("columns with missing values:", len(dq), "of", prev.shape[1])
print(dq.to_string())

print()
print("duplicate SK_ID_PREV:", int(prev.duplicated(subset=["SK_ID_PREV"]).sum()),
      "-> full-row duplicates impossible (unique application id)")

prev.to_parquet(OUT + "stage1_modeling.parquet", index=False)
check = pd.read_parquet(OUT + "stage1_modeling.parquet")
print("saved:", OUT + "stage1_modeling.parquet", check.shape,
      "| columns match:", list(check.columns) == list(prev.columns))
print("refusal rate in saved table: %.2f%%" % (check["REFUSED"].mean() * 100))
del check
gc.collect()
print("Free RAM (GB):", free_ram_gb())

columns with missing values: 137 of 156
                                   missing_pct                                                                 action
NAME_CASH_LOAN_PURPOSE                   94.89  missing after XNA/XAP removal; PURPOSE_STATED flag added where useful
CUST_COMMONAREA_AVG                      74.38           no customer match on the join; kept as NaN, model handles it
CUST_COMMONAREA_MEDI                     74.38           no customer match on the join; kept as NaN, model handles it
CUST_COMMONAREA_MODE                     74.38           no customer match on the join; kept as NaN, model handles it
CUST_NONLIVINGAPARTMENTS_AVG             74.07           no customer match on the join; kept as NaN, model handles it
CUST_NONLIVINGAPARTMENTS_MEDI            74.07           no customer match on the join; kept as NaN, model handles it
CUST_NONLIVINGAPARTMENTS_MODE            74.07           no customer match on the join; kept as NaN, model handles it
CUST_FONDKAPREMO

saved: ../outputs/stage1_modeling.parquet (1327459, 156) | columns match: True
refusal rate in saved table: 21.90%
Free RAM (GB): 0.52


In [14]:
# Cell N - star schema for Power BI: fact_applications + 4 dimension tables (spec Part 3)
# date dimension is derived from DAYS_DECISION because real calendar dates are hidden in this dataset
prev["DATE_KEY"] = np.ceil(-prev["DAYS_DECISION"] / 30.44).astype(np.int16)
dim_date = pd.DataFrame({"DATE_KEY": sorted(prev["DATE_KEY"].unique())})
dim_date["MONTHS_AGO"] = dim_date["DATE_KEY"]
dim_date["LAST_12M"] = (dim_date["MONTHS_AGO"] <= 12).astype(np.int8)
dim_date["HALF_YEAR"] = dim_date["MONTHS_AGO"].map(lambda m: "recent 6m" if m <= 6 else
    ("6-12m" if m <= 12 else ("12-24m" if m <= 24 else "older than 24m")))

# product dimension - integer key by tuple factorisation (key and dim guaranteed consistent)
prod_cols = ["NAME_CONTRACT_TYPE", "NAME_PRODUCT_TYPE", "NAME_PORTFOLIO",
             "NAME_GOODS_CATEGORY", "NAME_YIELD_GROUP", "NAME_PAYMENT_TYPE"]
codes, uniques = pd.MultiIndex.from_frame(prev[prod_cols]).factorize()
dim_product = uniques.to_frame(index=False)
dim_product.insert(0, "PRODUCT_KEY", np.arange(1, len(dim_product) + 1, dtype=np.int32))
prev["PRODUCT_KEY"] = codes + 1

chan_cols = ["CHANNEL_TYPE", "NAME_SELLER_INDUSTRY", "NAME_CLIENT_TYPE", "FLAG_LAST_APPL_PER_CONTRACT"]
codes, uniques = pd.MultiIndex.from_frame(prev[chan_cols]).factorize()
dim_channel = uniques.to_frame(index=False)
dim_channel.insert(0, "CHANNEL_KEY", np.arange(1, len(dim_channel) + 1, dtype=np.int32))
prev["CHANNEL_KEY"] = codes + 1

# client dimension - one row per customer (attributes identical across their applications)
client_cols = ["SK_ID_CURR", "AGE_YEARS", "EMPLOYED_YEARS", "CUST_CODE_GENDER",
               "CUST_NAME_CONTRACT_TYPE", "CUST_NAME_EDUCATION_TYPE", "CUST_NAME_FAMILY_STATUS",
               "CUST_NAME_INCOME_TYPE", "CUST_ORGANIZATION_TYPE", "CUST_CNT_CHILDREN",
               "CUST_CNT_FAM_MEMBERS", "CUST_EXT_SOURCE_1", "CUST_EXT_SOURCE_2", "CUST_EXT_SOURCE_3",
               "CUST_AMT_INCOME_TOTAL", "CUST_AMT_CREDIT", "CUST_FLAG_OWN_CAR",
               "CUST_FLAG_OWN_REALTY", "CUST_REGION_RATING_CLIENT"]
dim_client = prev[client_cols].drop_duplicates(subset=["SK_ID_CURR"]).reset_index(drop=True)

# fact table - foreign keys + the measures the dashboard needs
fact_cols = ["SK_ID_PREV", "SK_ID_CURR", "PRODUCT_KEY", "CHANNEL_KEY", "DATE_KEY",
             "REFUSED", "SPLIT", "AMT_APPLICATION", "AMT_DOWN_PAYMENT", "AMT_GOODS_PRICE",
             "AGE_BAND", "INCOME_BAND"]
fact = prev[fact_cols].copy()

tables = {"fact_applications": fact, "dim_client": dim_client, "dim_product": dim_product,
          "dim_channel": dim_channel, "dim_date": dim_date}
import os
total = 0
for name, t in tables.items():
    path = OUT + name + ".csv"
    t.to_csv(path, index=False)
    mb = os.path.getsize(path) / 1e6
    total += mb
    print("%-20s %s rows x %2d cols -> %.1f MB" % (name, format(len(t), ","), t.shape[1], mb))
print("all tables together: %.1f MB (Power BI handles this easily)" % total)
print("integrity: every fact key exists in its dimension ->",
      fact["PRODUCT_KEY"].isin(dim_product["PRODUCT_KEY"]).all()
      and fact["CHANNEL_KEY"].isin(dim_channel["CHANNEL_KEY"]).all()
      and fact["DATE_KEY"].isin(dim_date["DATE_KEY"]).all())
print("Free RAM (GB):", free_ram_gb())

fact_applications    1,327,459 rows x 12 cols -> 78.5 MB


dim_client           338,402 rows x 19 cols -> 44.9 MB
dim_product          435 rows x  7 cols -> 0.0 MB
dim_channel          210 rows x  5 cols -> 0.0 MB
dim_date             96 rows x  4 cols -> 0.0 MB
all tables together: 123.4 MB (Power BI handles this easily)
integrity: every fact key exists in its dimension -> True
Free RAM (GB): 1.84
